# VibeScent Week 2 Pipeline -- Harsh Agarwal
**P0 stages** (must-ship, ~60 min on A100) + **P1 stages** (spillover OK, ~2.5 additional hours)

Resumable via Drive checkpointing. GPU-tier auto-detection (A100/L4/T4).

In [ ]:
# === Stage 1: Setup ===
import subprocess, sys, os

# Clone repo if on Colab
if "COLAB_GPU" in os.environ:
    if not os.path.exists("/content/vibescent"):
        subprocess.run(["git", "clone", "-b", "Text_Processing",
                        "https://github.com/GavinGongTech/VibeScent.git", "/content/vibescent"], check=True)
    os.chdir("/content/vibescent")
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True, capture_output=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "notebooks/requirements.colab.txt"], check=True, capture_output=True)

# Mount Drive
try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    DRIVE_BASE = "/content/drive/MyDrive/vibescent/week2"
    os.makedirs(DRIVE_BASE, exist_ok=True)
except ImportError:
    DRIVE_BASE = "artifacts"  # local fallback

# Load optional API keys
try:
    GEMINI_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_KEY = os.environ.get("GEMINI_API_KEY")
try:
    VOYAGE_KEY = userdata.get("VOYAGE_API_KEY")
except Exception:
    VOYAGE_KEY = os.environ.get("VOYAGE_API_KEY")

if GEMINI_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_KEY
if VOYAGE_KEY:
    os.environ["VOYAGE_API_KEY"] = VOYAGE_KEY

print(f"Drive base: {DRIVE_BASE}")
print(f"Gemini key: {'set' if GEMINI_KEY else 'NOT SET'}")
print(f"Voyage key: {'set' if VOYAGE_KEY else 'NOT SET'}")

In [ ]:
# === Stage 1b: GPU Detection + Pipeline Config ===
import torch
from vibescents.week2_pipeline import detect_gpu_tier, check_disk_space

PIPELINE_VERSION = "v1"

try:
    GPU_TIER = detect_gpu_tier()
except RuntimeError:
    GPU_TIER = "CPU"

USE_LOCAL_LLM = GPU_TIER == "A100"
USE_LOCAL_MM = GPU_TIER in ("A100", "L4")
USE_INT8_EMBEDDER = GPU_TIER == "T4"

print(f"GPU Tier: {GPU_TIER}")
print(f"Local LLM: {USE_LOCAL_LLM} | Local MM: {USE_LOCAL_MM} | Int8 Embedder: {USE_INT8_EMBEDDER}")

if "COLAB_GPU" in os.environ:
    check_disk_space("/content")

In [ ]:
# === Stage 2: Sanity Check ===
import pandas as pd
from pathlib import Path

df = pd.read_csv("data/vibescent_unified.csv")
assert len(df) == 35889, f"Expected 35889 rows, got {len(df)}"
assert Path("examples/occasions.json").exists(), "occasions.json missing"
assert Path("examples/benchmark_briefs.json").exists(), "benchmark_briefs.json missing"

# Spot-check 5 rows
sample = df.sample(5, random_state=42)[["name", "brand", "top_notes"]].to_string()
print(f"Corpus: {len(df)} rows")
print(f"Columns: {list(df.columns)}")
print(f"\nSample rows:\n{sample}")

# Check embedding_text column exists
if "embedding_text" not in df.columns:
    print("WARNING: embedding_text column missing -- will need to construct it")

In [ ]:
# === Stage 3: Load Qwen3-Embedding-8B ===
from vibescents.week2_pipeline import ensure_model_loaded, stage_complete, mark_model_unloaded
import numpy as np
from pathlib import Path

ARTIFACTS_RAW = Path(DRIVE_BASE) / "fragrance_raw_full"

if stage_complete("embed_raw", ARTIFACTS_RAW, PIPELINE_VERSION):
    print("Stage 3: embeddings already complete, skipping model load")
    embedder = None
else:
    from sentence_transformers import SentenceTransformer

    model_kwargs = {"torch_dtype": torch.float16}
    if USE_INT8_EMBEDDER:
        model_kwargs = {"load_in_8bit": True}

    embedder = SentenceTransformer(
        "Qwen/Qwen3-Embedding-8B",
        trust_remote_code=True,
        model_kwargs=model_kwargs,
    )
    ensure_model_loaded("qwen3-embed-8b", lambda: None)  # track state

    # Pre-flight dim assertion
    test_emb = embedder.encode(["test sentence one", "test sentence two"], normalize_embeddings=True)
    assert test_emb.shape[1] == 4096, f"Expected 4096 dims, got {test_emb.shape[1]}"
    print(f"Pre-flight OK: Qwen3-Embedding-8B loaded, output dim={test_emb.shape[1]}")

    if "COLAB_GPU" in os.environ:
        check_disk_space("/content")

In [ ]:
# === Stage 4: Embed Full 35,889 Corpus at 1024 dim ===
from vibescents.week2_pipeline import (
    embed_corpus, embed_corpus_resume, embedding_sanity_check, write_manifest
)
import subprocess

ARTIFACTS_RAW = Path(DRIVE_BASE) / "fragrance_raw_full"

if stage_complete("embed_raw", ARTIFACTS_RAW, PIPELINE_VERSION):
    print("Stage 4: loading cached embeddings")
    raw_embeddings = np.load(ARTIFACTS_RAW / "embeddings.npy")
    print(f"Loaded: {raw_embeddings.shape}")
else:
    # Determine text column
    text_col = "embedding_text" if "embedding_text" in df.columns else "retrieval_text"
    if text_col not in df.columns:
        # Construct from available columns
        texts = (df["name"].fillna("") + " " + df.get("brand", pd.Series([""] * len(df))).fillna("") +
                 " " + df.get("main_accords", pd.Series([""] * len(df))).fillna("") +
                 " " + df.get("top_notes", pd.Series([""] * len(df))).fillna("") +
                 " " + df.get("middle_notes", pd.Series([""] * len(df))).fillna("") +
                 " " + df.get("base_notes", pd.Series([""] * len(df))).fillna("")).tolist()
    else:
        texts = df[text_col].fillna("").tolist()

    # Check for resume
    CKPT_DIR = Path(DRIVE_BASE) / "embed_raw_ckpt"
    partial, resume_batch = embed_corpus_resume(CKPT_DIR)
    if partial is not None:
        print(f"Resuming from batch {resume_batch}, {partial.shape[0]} rows already done")

    # Create a wrapper that calls SentenceTransformer.encode
    class STWrapper:
        def __init__(self, model):
            self._model = model
        def embed_texts(self, batch):
            return self._model.encode(batch, normalize_embeddings=True, show_progress_bar=False)

    raw_embeddings = embed_corpus(
        texts, STWrapper(embedder), dim=1024, batch_size=32,
        checkpoint_dir=CKPT_DIR, resume_batch=resume_batch,
    )

    # If we resumed, prepend the partial
    if partial is not None:
        raw_embeddings = np.vstack([partial, raw_embeddings])

    # Sanity checks
    assert raw_embeddings.shape == (35889, 1024), f"Shape mismatch: {raw_embeddings.shape}"
    embedding_sanity_check(raw_embeddings)

    # Save
    ARTIFACTS_RAW.mkdir(parents=True, exist_ok=True)
    np.save(ARTIFACTS_RAW / "embeddings.npy", raw_embeddings)

    commit_sha = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    write_manifest(
        ARTIFACTS_RAW / "manifest.json",
        model="Qwen/Qwen3-Embedding-8B",
        commit_sha=commit_sha,
        row_count=raw_embeddings.shape[0],
        dim=raw_embeddings.shape[1],
        pipeline_version=PIPELINE_VERSION,
    )
    print(f"Saved: {raw_embeddings.shape} to {ARTIFACTS_RAW}")

    if "COLAB_GPU" in os.environ:
        check_disk_space("/content")

In [ ]:
# === Stage 5: Embed 8 Occasions ===
import json

ARTIFACTS_OCC = Path(DRIVE_BASE) / "occasions"

if stage_complete("embed_occ", ARTIFACTS_OCC, PIPELINE_VERSION):
    print("Stage 5: loading cached occasion embeddings")
    occ_embeddings = np.load(ARTIFACTS_OCC / "embeddings.npy")
else:
    with open("examples/occasions.json") as f:
        occasions = json.load(f)

    occ_texts = [occ["text"] if isinstance(occ, dict) else str(occ) for occ in occasions]

    class STWrapper:
        def __init__(self, model):
            self._model = model
        def embed_texts(self, batch):
            return self._model.encode(batch, normalize_embeddings=True, show_progress_bar=False)

    occ_embeddings = embed_corpus(occ_texts, STWrapper(embedder), dim=1024, batch_size=8)

    ARTIFACTS_OCC.mkdir(parents=True, exist_ok=True)
    np.save(ARTIFACTS_OCC / "embeddings.npy", occ_embeddings)
    commit_sha = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    write_manifest(
        ARTIFACTS_OCC / "manifest.json",
        model="Qwen/Qwen3-Embedding-8B",
        commit_sha=commit_sha,
        row_count=occ_embeddings.shape[0],
        dim=occ_embeddings.shape[1],
        pipeline_version=PIPELINE_VERSION,
    )
    print(f"Saved: {occ_embeddings.shape} to {ARTIFACTS_OCC}")

print(f"Occasions: {occ_embeddings.shape}")

In [ ]:
# === Stage 6: Unload Embedder, Load Qwen3.5-27B ===
from vibescents.week2_pipeline import smoke_test_enrichment

# Unload embedder
if embedder is not None:
    del embedder
    torch.cuda.empty_cache()
    mark_model_unloaded()
    print("Embedder unloaded")

if USE_LOCAL_LLM:
    # Try loading Qwen3.5-27B via the existing enrich.py Qwen path
    from vibescents.enrich import QwenOutlinesEnrichmentClient
    try:
        llm_client = QwenOutlinesEnrichmentClient(model_name="Qwen/Qwen3.5-27B-GPTQ-Int4")
        ensure_model_loaded("qwen3.5-27b", lambda: None)
        print("Qwen3.5-27B-GPTQ-Int4 loaded via Outlines")
    except Exception as e:
        print(f"Outlines failed: {e}")
        print("Falling back to Gemini API")
        USE_LOCAL_LLM = False

if not USE_LOCAL_LLM:
    from vibescents.enrich import GeminiEnrichmentClient
    llm_client = GeminiEnrichmentClient()
    ensure_model_loaded("gemini-api", lambda: None)
    print("Using Gemini API for enrichment")

# 1-row smoke test
sample_row = df.iloc[0]
try:
    smoke_test_enrichment(llm_client, sample_row)
    print("Smoke test PASSED")
except Exception as e:
    print(f"Smoke test FAILED: {e}")
    print("Proceeding anyway -- individual row failures will be caught by the retry chain")

if "COLAB_GPU" in os.environ:
    check_disk_space("/content")

In [ ]:
# === Stage 7: Generate 20 Benchmark Labels (3 runs) ===
from vibescents.benchmark import BenchmarkGenerator, consolidate_case_drafts

BENCHMARK_PATH = Path(DRIVE_BASE) / "benchmark_cases.json"

if BENCHMARK_PATH.exists():
    with open(BENCHMARK_PATH) as f:
        benchmark_cases = json.load(f)
    print(f"Loaded {len(benchmark_cases)} cached benchmark cases")
else:
    benchmark_provider = "qwen" if USE_LOCAL_LLM else "gemini"
    generator = BenchmarkGenerator(provider=benchmark_provider)

    with open("examples/benchmark_briefs.json") as f:
        briefs = json.load(f)

    benchmark_cases = []
    for brief_obj in briefs:
        case_id = brief_obj["case_id"]
        brief_text = brief_obj["brief"]
        print(f"  Generating labels for {case_id}...")
        drafts = generator.generate_case_labels(case_id=case_id, brief=brief_text, runs=3)
        consolidated = consolidate_case_drafts(drafts)
        benchmark_cases.append(consolidated.model_dump())

    with open(BENCHMARK_PATH, "w") as f:
        json.dump(benchmark_cases, f, indent=2)

    valid = sum(1 for c in benchmark_cases if c.get("confidence", 0) >= 0.6)
    print(f"Generated {len(benchmark_cases)} cases, {valid} with confidence >= 0.6")

print(f"Benchmark cases: {len(benchmark_cases)}")

In [ ]:
# === Stage 8: Enrich TIER C (500-row sample) ===
from vibescents.enrich import enrich_dataframe, build_retrieval_text
from vibescents.week2_pipeline import validate_enrichment

TIER_C_PATH = Path(DRIVE_BASE) / "vibescent_enriched_500_v2.csv"

if TIER_C_PATH.exists():
    enriched_c = pd.read_csv(TIER_C_PATH)
    print(f"Loaded cached TIER C: {len(enriched_c)} rows")
else:
    df_500 = pd.read_csv("data/vibescent_500.csv") if Path("data/vibescent_500.csv").exists() else df.head(500).copy()

    provider = "qwen" if USE_LOCAL_LLM else "gemini"
    enriched_c = enrich_dataframe(
        df_500,
        provider=provider,
        max_rows=500,
        checkpoint_path=str(TIER_C_PATH) + ".ckpt",
        failures_path=str(TIER_C_PATH) + ".failures.jsonl",
        batch_size=16,
    )
    enriched_c = build_retrieval_text(enriched_c)
    enriched_c.to_csv(TIER_C_PATH, index=False)
    print(f"Saved TIER C: {len(enriched_c)} rows to {TIER_C_PATH}")

# Quality gate
try:
    validate_enrichment(enriched_c)
except AssertionError as e:
    print(f"WARNING: {e}")
    print("Check failures log for details")

In [ ]:
# === Stage 9: Retrieval Comparison (TIER A raw) + Report ===
from vibescents.similarity import cosine_similarity_matrix, top_k_indices

# Load occasion embeddings
occ_emb = np.load(Path(DRIVE_BASE) / "occasions" / "embeddings.npy")
raw_emb = np.load(Path(DRIVE_BASE) / "fragrance_raw_full" / "embeddings.npy")

# Cosine similarity: occasions vs raw corpus
sim_matrix = cosine_similarity_matrix(occ_emb, raw_emb)

# Top-5 per occasion
with open("examples/occasions.json") as f:
    occasions = json.load(f)

print("=== TIER A: Raw Corpus Top-5 per Occasion ===\n")
for i, occ in enumerate(occasions):
    occ_text = occ["text"] if isinstance(occ, dict) else str(occ)
    top5_idx = top_k_indices(sim_matrix[i], k=5)
    print(f"Occasion: {occ_text}")
    for rank, idx in enumerate(top5_idx, 1):
        name = df.iloc[idx].get("name", "?")
        brand = df.iloc[idx].get("brand", "?")
        score = sim_matrix[i, idx]
        print(f"  {rank}. {brand} -- {name} (score: {score:.4f})")
    print()

print("P0 STAGES COMPLETE")
print(f"Artifacts saved to: {DRIVE_BASE}")

---
## P1 Stages (spillover OK, ~2.5 additional hours)
Stages 10-21 below. Run if time allows in this Colab session.

In [ ]:
# === Stage 10: Enrich TIER B (top-2K rows) ===
from vibescents.week2_pipeline import select_tier_b

TIER_B_PATH = Path(DRIVE_BASE) / "vibescent_enriched_2k_v2.csv"

if TIER_B_PATH.exists():
    enriched_b = pd.read_csv(TIER_B_PATH)
    print(f"Loaded cached TIER B: {len(enriched_b)} rows")
else:
    tier_b_df = select_tier_b(df, target_size=2000, min_size=500)
    
    provider = "qwen" if USE_LOCAL_LLM else "gemini"
    enriched_b = enrich_dataframe(
        tier_b_df,
        provider=provider,
        max_rows=len(tier_b_df),
        checkpoint_path=str(TIER_B_PATH) + ".ckpt",
        failures_path=str(TIER_B_PATH) + ".failures.jsonl",
        batch_size=16,
    )
    enriched_b = build_retrieval_text(enriched_b)
    enriched_b.to_csv(TIER_B_PATH, index=False)
    print(f"Saved TIER B: {len(enriched_b)} rows to {TIER_B_PATH}")

validate_enrichment(enriched_b)

In [ ]:
# === Stage 11: Rebuild retrieval_text for enriched tiers ===
# Already done during enrichment via build_retrieval_text()
# Re-run here in case we loaded from cache without retrieval_text
enriched_b = build_retrieval_text(enriched_b)
enriched_c = build_retrieval_text(enriched_c)
print(f"TIER B retrieval_text sample:\n{enriched_b['retrieval_text'].iloc[0][:200]}...")
print(f"\nTIER C retrieval_text sample:\n{enriched_c['retrieval_text'].iloc[0][:200]}...")

In [ ]:
# === Stage 12: Generate display_text for TIER B (stopgap for Karan) ===
# This stage uses the LLM to generate 1-sentence tasting notes.
# Skipping in this session if time is tight — display_text can be generated later.
# Uncomment below to enable:
#
# DISPLAY_TEXT_PATH = Path(DRIVE_BASE) / "display_texts.csv"
# if not DISPLAY_TEXT_PATH.exists():
#     # Would need a separate prompt + schema for display_text generation
#     pass
print("Stage 12: display_text generation SKIPPED (optional, run separately if needed)")

In [ ]:
# === Stage 13: Unload LLM, Reload Qwen3-Embedding-8B ===
if "llm_client" in dir():
    del llm_client
torch.cuda.empty_cache()
mark_model_unloaded()

from sentence_transformers import SentenceTransformer

model_kwargs = {"torch_dtype": torch.float16}
if USE_INT8_EMBEDDER:
    model_kwargs = {"load_in_8bit": True}

embedder = SentenceTransformer(
    "Qwen/Qwen3-Embedding-8B",
    trust_remote_code=True,
    model_kwargs=model_kwargs,
)
ensure_model_loaded("qwen3-embed-8b", lambda: None)
print("Qwen3-Embedding-8B reloaded for enriched embedding")

if "COLAB_GPU" in os.environ:
    check_disk_space("/content")

In [ ]:
# === Stage 14: Embed Enriched TIER B + TIER C ===
ARTIFACTS_ENR_B = Path(DRIVE_BASE) / "fragrance_enriched_2k"
ARTIFACTS_ENR_C = Path(DRIVE_BASE) / "fragrance_enriched_500"

class STWrapper:
    def __init__(self, model):
        self._model = model
    def embed_texts(self, batch):
        return self._model.encode(batch, normalize_embeddings=True, show_progress_bar=False)

wrapper = STWrapper(embedder)

# TIER B enriched
if stage_complete("embed_enr_b", ARTIFACTS_ENR_B, PIPELINE_VERSION):
    enriched_b_emb = np.load(ARTIFACTS_ENR_B / "embeddings.npy")
    print(f"Loaded cached TIER B embeddings: {enriched_b_emb.shape}")
else:
    texts_b = enriched_b["retrieval_text"].fillna("").tolist()
    enriched_b_emb = embed_corpus(texts_b, wrapper, dim=1024, batch_size=32)
    ARTIFACTS_ENR_B.mkdir(parents=True, exist_ok=True)
    np.save(ARTIFACTS_ENR_B / "embeddings.npy", enriched_b_emb)
    commit_sha = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    write_manifest(ARTIFACTS_ENR_B / "manifest.json", model="Qwen/Qwen3-Embedding-8B",
                   commit_sha=commit_sha, row_count=enriched_b_emb.shape[0],
                   dim=enriched_b_emb.shape[1], pipeline_version=PIPELINE_VERSION)
    print(f"Saved TIER B enriched embeddings: {enriched_b_emb.shape}")

# TIER C enriched
if stage_complete("embed_enr_c", ARTIFACTS_ENR_C, PIPELINE_VERSION):
    enriched_c_emb = np.load(ARTIFACTS_ENR_C / "embeddings.npy")
    print(f"Loaded cached TIER C embeddings: {enriched_c_emb.shape}")
else:
    texts_c = enriched_c["retrieval_text"].fillna("").tolist()
    enriched_c_emb = embed_corpus(texts_c, wrapper, dim=1024, batch_size=32)
    ARTIFACTS_ENR_C.mkdir(parents=True, exist_ok=True)
    np.save(ARTIFACTS_ENR_C / "embeddings.npy", enriched_c_emb)
    commit_sha = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    write_manifest(ARTIFACTS_ENR_C / "manifest.json", model="Qwen/Qwen3-Embedding-8B",
                   commit_sha=commit_sha, row_count=enriched_c_emb.shape[0],
                   dim=enriched_c_emb.shape[1], pipeline_version=PIPELINE_VERSION)
    print(f"Saved TIER C enriched embeddings: {enriched_c_emb.shape}")

embedding_sanity_check(enriched_b_emb)
print("Enriched embedding sanity check passed")

In [ ]:
# === Stage 15: RAW vs ENRICHED Retrieval Comparison ===
occ_emb = np.load(Path(DRIVE_BASE) / "occasions" / "embeddings.npy")
raw_emb = np.load(Path(DRIVE_BASE) / "fragrance_raw_full" / "embeddings.npy")

# For fair comparison, subset raw embeddings to the same rows as TIER C
if Path("data/vibescent_500.csv").exists():
    df_500 = pd.read_csv("data/vibescent_500.csv")
else:
    df_500 = df.head(500)

# Raw embeddings for the 500 rows (indices 0-499 from the full corpus)
raw_500_emb = raw_emb[:500]

sim_raw = cosine_similarity_matrix(occ_emb, raw_500_emb)
sim_enr = cosine_similarity_matrix(occ_emb, enriched_c_emb)

with open("examples/occasions.json") as f:
    occasions = json.load(f)

comparison_lines = []
comparison_lines.append("=== TIER C: RAW vs ENRICHED Top-5 per Occasion ===\n")

different_count = 0
for i, occ in enumerate(occasions):
    occ_text = occ["text"] if isinstance(occ, dict) else str(occ)
    top5_raw = set(top_k_indices(sim_raw[i], k=5).tolist())
    top5_enr = set(top_k_indices(sim_enr[i], k=5).tolist())
    overlap = len(top5_raw & top5_enr)
    if overlap < 5:
        different_count += 1
    line = f"Occasion: {occ_text} | Overlap: {overlap}/5 | Different: {5 - overlap}"
    comparison_lines.append(line)

comparison_lines.append(f"\nOccasions with >=1 different result: {different_count}/{len(occasions)}")

comparison_text = "\n".join(comparison_lines)
print(comparison_text)

# Save
comp_path = Path(DRIVE_BASE) / "retrieval_comparison.txt"
comp_path.write_text(comparison_text)
print(f"\nSaved to {comp_path}")

In [ ]:
# === Stage 16: Load Qwen3-VL-Embedding-8B ===
if not USE_LOCAL_MM:
    print("Stage 16: SKIPPED (no local multimodal on this GPU tier)")
    vl_embedder = None
else:
    del embedder
    torch.cuda.empty_cache()
    mark_model_unloaded()
    
    from vibescents.qwen3_vl_embedding import Qwen3VLMultimodalEmbedder
    vl_embedder = Qwen3VLMultimodalEmbedder()
    ensure_model_loaded("qwen3-vl-embed-8b", lambda: None)
    print("Qwen3-VL-Embedding-8B loaded")
    
    if "COLAB_GPU" in os.environ:
        check_disk_space("/content")

In [ ]:
# === Stage 17: Multimodal Embed TIER B Documents ===
ARTIFACTS_MM = Path(DRIVE_BASE) / "multimodal_2k"

if vl_embedder is None:
    print("Stage 17: SKIPPED (no multimodal embedder)")
elif stage_complete("mm_embed_b", ARTIFACTS_MM, PIPELINE_VERSION):
    mm_doc_emb = np.load(ARTIFACTS_MM / "doc_embeddings.npy")
    print(f"Loaded cached multimodal doc embeddings: {mm_doc_emb.shape}")
else:
    texts_b = enriched_b["retrieval_text"].fillna("").tolist()
    # Embed in batches of 8 (VL model is heavier)
    mm_parts = []
    for start in range(0, len(texts_b), 8):
        batch = texts_b[start:start+8]
        emb = vl_embedder.embed_texts(batch)
        mm_parts.append(emb[:, :1024])  # Matryoshka truncate
        if (start // 8 + 1) % 50 == 0:
            print(f"  MM embed batch {start//8 + 1}/{(len(texts_b)+7)//8}")
    
    mm_doc_emb = np.vstack(mm_parts).astype(np.float32)
    norms = np.linalg.norm(mm_doc_emb, axis=1, keepdims=True)
    mm_doc_emb = mm_doc_emb / np.where(norms == 0, 1, norms)
    
    ARTIFACTS_MM.mkdir(parents=True, exist_ok=True)
    np.save(ARTIFACTS_MM / "doc_embeddings.npy", mm_doc_emb)
    commit_sha = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    write_manifest(ARTIFACTS_MM / "manifest.json", model="Qwen/Qwen3-VL-Embedding-8B",
                   commit_sha=commit_sha, row_count=mm_doc_emb.shape[0],
                   dim=mm_doc_emb.shape[1], pipeline_version=PIPELINE_VERSION)
    print(f"Saved multimodal doc embeddings: {mm_doc_emb.shape}")

In [ ]:
# === Stage 18: Multimodal Query Probes ===
if vl_embedder is None:
    print("Stage 18: SKIPPED (no multimodal embedder)")
else:
    # Use occasion texts as queries (no outfit images bundled yet)
    with open("examples/occasions.json") as f:
        occasions = json.load(f)
    
    query_texts = [occ["text"] if isinstance(occ, dict) else str(occ) for occ in occasions]
    query_emb = vl_embedder.embed_texts(query_texts)
    query_emb = query_emb[:, :1024].astype(np.float32)
    norms = np.linalg.norm(query_emb, axis=1, keepdims=True)
    query_emb = query_emb / np.where(norms == 0, 1, norms)
    
    mm_sim = cosine_similarity_matrix(query_emb, mm_doc_emb)
    
    print("=== Multimodal Text-Only Query Top-5 ===\n")
    for i, occ in enumerate(occasions):
        occ_text = occ["text"] if isinstance(occ, dict) else str(occ)
        top5 = top_k_indices(mm_sim[i], k=5)
        print(f"Query: {occ_text}")
        for rank, idx in enumerate(top5, 1):
            name = enriched_b.iloc[idx].get("name", "?")
            brand = enriched_b.iloc[idx].get("brand", "?")
            print(f"  {rank}. {brand} -- {name} (score: {mm_sim[i, idx]:.4f})")
        print()

In [ ]:
# === Stage 19: Sanity Checks ===
from vibescents.week2_pipeline import read_manifest

print("=== Artifact Sanity Checks ===\n")
for name, path in [
    ("Raw full", Path(DRIVE_BASE) / "fragrance_raw_full"),
    ("Occasions", Path(DRIVE_BASE) / "occasions"),
    ("Enriched 2K", Path(DRIVE_BASE) / "fragrance_enriched_2k"),
    ("Enriched 500", Path(DRIVE_BASE) / "fragrance_enriched_500"),
]:
    manifest_path = path / "manifest.json"
    if manifest_path.exists():
        m = read_manifest(manifest_path)
        emb_path = path / "embeddings.npy"
        if emb_path.exists():
            shape = np.load(emb_path, mmap_mode="r").shape
            ok = shape[0] == m["row_count"] and shape[1] == m["dim"]
            status = "OK" if ok else f"MISMATCH (file={shape}, manifest=({m['row_count']}, {m['dim']}))"
        else:
            status = "NO EMBEDDINGS FILE"
        print(f"  {name}: {status} | version={m['pipeline_version']}")
    else:
        print(f"  {name}: NO MANIFEST")

print("\nAll sanity checks complete.")

In [ ]:
# === Stage 20: Report Writer ===
# Populate results/week2_report.md if it exists
report_path = Path("results/week2_report.md")
if report_path.exists():
    content = report_path.read_text()
    # Fix stray trailing backtick if present
    if content.rstrip().endswith("`") and not content.rstrip().endswith("```"):
        content = content.rstrip()[:-1] + "\n"
        report_path.write_text(content)
        print("Fixed trailing backtick in week2_report.md")
    else:
        print("No trailing backtick issue found")
else:
    print("results/week2_report.md not found -- create it manually or in the next session")

print("Stage 20: Report updates should be done manually based on the artifacts above")

In [ ]:
# === Stage 21: Triple Artifact Sink ===
import shutil

# 1. Drive — already saved throughout (DRIVE_BASE)
print(f"Drive artifacts at: {DRIVE_BASE}")

# 2. Repo artifacts — copy small files only
repo_artifacts = Path("artifacts")
repo_artifacts.mkdir(exist_ok=True)

# Copy occasion embeddings (small, < 1 MB)
occ_src = Path(DRIVE_BASE) / "occasions"
occ_dst = repo_artifacts / "occasions"
if occ_src.exists():
    occ_dst.mkdir(parents=True, exist_ok=True)
    for f in occ_src.iterdir():
        if f.stat().st_size < 20_000_000:  # < 20 MB
            shutil.copy2(f, occ_dst / f.name)
    print(f"Copied occasions artifacts to {occ_dst}")

# Copy benchmark cases
bm_src = Path(DRIVE_BASE) / "benchmark_cases.json"
if bm_src.exists():
    shutil.copy2(bm_src, repo_artifacts / "benchmark_cases.json")
    print("Copied benchmark_cases.json to artifacts/")

# Copy retrieval comparison
comp_src = Path(DRIVE_BASE) / "retrieval_comparison.txt"
if comp_src.exists():
    shutil.copy2(comp_src, repo_artifacts / "retrieval_comparison.txt")
    print("Copied retrieval_comparison.txt to artifacts/")

# 3. Generate git add snippet (exclude large files)
print("\n=== Git Add Snippet (excludes .npy > 20 MB) ===")
print("git add artifacts/occasions/ artifacts/benchmark_cases.json artifacts/retrieval_comparison.txt")
print("git add notebooks/harsh_week2_pipeline.ipynb")
print("git add src/vibescents/week2_pipeline.py tests/test_week2_pipeline.py")

# 4. Colab download (if running on Colab)
try:
    from google.colab import files
    # Bundle small artifacts for download
    print("\nDownloading key artifacts...")
    if Path(DRIVE_BASE, "benchmark_cases.json").exists():
        files.download(str(Path(DRIVE_BASE) / "benchmark_cases.json"))
except ImportError:
    pass

print("\n=== WEEK 2 PIPELINE COMPLETE ===")
print(f"All artifacts saved to: {DRIVE_BASE}")